In [ ]:
import pandas as pd

output_file = "exact_and_partial_match_for_all_models_drug_extraction.txt"

# Helper to write both to console & file
def write_results(text):
    print(text)
    with open(output_file, "a") as f:
        f.write(text + "\n")


# ------------------------------------------
# Utility: overlap detection
# ------------------------------------------

def spans_overlap(span1, span2):
    s1, e1 = span1
    s2, e2 = span2
    return not (e1 < s2 or e2 < s1)

def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2*precision*recall / (precision + recall) if (precision+recall)>0 else 0.0
    return precision, recall, f1

# Convert list of strings → artificial spans
def to_spans(entities):
    spans = []
    start = 0
    for ent in entities:
        length = len(ent)
        spans.append(((start, start + length - 1), ent))
        start += length + 1
    return spans

# ------------------------------------------
# i2b2: strict & relaxed matching
# ------------------------------------------

def score_entities(pred_list, gold_list, mode="strict"):
    pred_spans = to_spans(pred_list)
    gold_spans = to_spans(gold_list)

    matched_gold = set()
    TP = FP = FN = 0

    for (p_span, p_text) in pred_spans:
        found = False
        for i, (g_span, g_text) in enumerate(gold_spans):
            if i in matched_gold:
                continue

            if mode == "strict":
                if p_text == g_text and p_span == g_span:
                    TP += 1
                    matched_gold.add(i)
                    found = True
                    break

            elif mode == "relaxed":
                if p_text == g_text and spans_overlap(p_span, g_span):
                    TP += 1
                    matched_gold.add(i)
                    found = True
                    break

        if not found:
            FP += 1

    FN = len(gold_spans) - len(matched_gold)
    return TP, FP, FN

# ------------------------------------------
# Evaluate each entity type
# ------------------------------------------

def evaluate_entity_type(df, pred_col, gold_col):
    TP_s = FP_s = FN_s = 0
    TP_r = FP_r = FN_r = 0

    for _, row in df.iterrows():
        pred = row[pred_col]
        gold = row[gold_col]

        pred = pred.split(",") if isinstance(pred, str) else []
        gold = gold.split(",") if isinstance(gold, str) else []

        pred = [x.strip() for x in pred if x.strip()]
        gold = [x.strip() for x in gold if x.strip()]

        tp, fp, fn = score_entities(pred, gold, "strict")
        TP_s += tp; FP_s += fp; FN_s += fn

        tp, fp, fn = score_entities(pred, gold, "relaxed")
        TP_r += tp; FP_r += fp; FN_r += fn

    strict = compute_metrics(TP_s, FP_s, FN_s)
    relaxed = compute_metrics(TP_r, FP_r, FN_r)

    return {"strict": strict, "relaxed": relaxed}


# -----------------------------------------------------------
# PROCESS ALL 4 FILES (results go to results.txt)
# -----------------------------------------------------------

open(output_file, "w").close()   # clear file at start

datasets = [
    ("GEMMA", "./data/LLaVA-Med/Extracted_Drugs_GEMMA_ADRs_151125.csv"),
    ("LLAMA", "./data/LLaVA-Med/Extracted_Drugs_LLAMA_ADRs_081125.csv"),
    ("MEDLLAMA", "./data/LLaVA-Med/Extracted_Drugs_MEDLLAMA_ADRs_151125.csv"),
    ("QWEN", "./data/LLaVA-Med/Extracted_Drugs_QWEN_ADRs_151125.csv"),
]

for name, path in datasets:
    write_results(f"\n====================== {name} RESULTS ======================\n")

    df = pd.read_csv(path)

    # Drug Name Extraction
    drug_results = evaluate_entity_type(df, pred_col="drug_names", gold_col="Drug Name")

    # ADR Extraction
    adr_results = evaluate_entity_type(df, pred_col="adverse_effects", gold_col="Side/Harmful effects")

    write_results("==== DRUG NAME RESULTS ====")
    write_results(f"Strict  (P,R,F1): {drug_results['strict']}")
    write_results(f"Relaxed (P,R,F1): {drug_results['relaxed']}")

    write_results("\n==== ADR RESULTS ====")
    write_results(f"Strict  (P,R,F1): {adr_results['strict']}")
    write_results(f"Relaxed (P,R,F1): {adr_results['relaxed']}")
    write_results("\n")
